# 06 — **AyurGenix curated rows** for a **second Vector Search index** (warehouse-free RAG)

Builds `curated_rows_for_vector` with:

- **`curated_pk`** — stable `sha2(...)` primary key for Delta Sync.
- **`embedding_text`** — **symptom-heavy** text: the `Symptoms` column is repeated multiple times at the top (with explicit labels) so the managed embedding model **up-weights** symptom overlap vs. disease names alone.
- **Structured payload columns** — returned at query time (do not pass `embedding_text` to the LLM in the app — only for indexing).

**Prerequisite:** `ayurgenix_curated` (notebook **02**).

**After this notebook:** create a **second** Mosaic AI Vector Search **Delta Sync** index:

- Source: `ayurveda_lakehouse.ayurgenix.curated_rows_for_vector`
- Primary key: `curated_pk`
- Embedding source column: `embedding_text`
- Managed embeddings: `databricks-qwen3-embedding-0-6b` (or your workspace’s multilingual FM)

Set on the Databricks App: `VECTOR_SEARCH_INDEX_CURATED` = full UC index name (same endpoint as PDF index is fine).


In [ ]:
%sql
USE CATALOG ayurveda_lakehouse;
USE SCHEMA ayurgenix;


## Step 1 — Materialise `curated_rows_for_vector` (symptom-weighted `embedding_text`)

Symptom emphasis: **four** consecutive blocks labelled `Symptoms` plus a short `SYMPTOM_KEYWORDS` line (same text) before the rest of the clinical profile. Remaining columns are appended once for factual grounding.


In [ ]:
%sql
CREATE OR REPLACE TABLE ayurveda_lakehouse.ayurgenix.curated_rows_for_vector AS
SELECT
  sha2(
    concat_ws(
      '||',
      coalesce(Disease, ''),
      coalesce(Symptoms, ''),
      coalesce(Ayurvedic_Herbs, ''),
      coalesce(Formulation, ''),
      coalesce(Doshas, '')
    ),
    256
  ) AS curated_pk,
  Disease,
  Symptoms,
  Hindi_Name,
  Marathi_Name,
  Ayurvedic_Herbs,
  Formulation,
  Doshas,
  Herbal_Alternative_Remedies,
  Diet_and_Lifestyle_Recommendations,
  Patient_Recommendations,
  Diagnosis_Tests,
  Symptom_Severity,
  concat(
    '[RETRIEVAL_PRIORITY:USER_SYMPTOMS_MATCH_AGAINST_BLOCKS_BELOW]\n',
    'SYMPTOM_BLOCK_1:\n', coalesce(Symptoms, ''), '\n\n',
    'SYMPTOM_BLOCK_2:\n', coalesce(Symptoms, ''), '\n\n',
    'SYMPTOM_BLOCK_3:\n', coalesce(Symptoms, ''), '\n\n',
    'SYMPTOM_BLOCK_4:\n', coalesce(Symptoms, ''), '\n\n',
    'SYMPTOM_KEYWORDS_COMPACT:\n', coalesce(Symptoms, ''), '\n\n',
    '---STRUCTURED_CONTEXT---\n',
    'Disease_label: ', coalesce(Disease, ''), '\n',
    'Hindi_Name: ', coalesce(Hindi_Name, ''), '\n',
    'Marathi_Name: ', coalesce(Marathi_Name, ''), '\n',
    'Diagnosis_Tests: ', coalesce(Diagnosis_Tests, ''), '\n',
    'Symptom_Severity: ', coalesce(Symptom_Severity, ''), '\n',
    'Doshas: ', coalesce(Doshas, ''), '\n',
    'Ayurvedic_Herbs: ', coalesce(Ayurvedic_Herbs, ''), '\n',
    'Formulation: ', coalesce(Formulation, ''), '\n',
    'Herbal_Alternative_Remedies: ', coalesce(Herbal_Alternative_Remedies, ''), '\n',
    'Diet_and_Lifestyle_Recommendations: ', coalesce(Diet_and_Lifestyle_Recommendations, ''), '\n',
    'Patient_Recommendations: ', coalesce(Patient_Recommendations, '')
  ) AS embedding_text
FROM ayurveda_lakehouse.ayurgenix.ayurgenix_curated;

SELECT COUNT(*) AS curated_vector_rows FROM ayurveda_lakehouse.ayurgenix.curated_rows_for_vector;


## Step 2 — (Optional) Create **curated** Vector Search index (Python SDK)

Use the **same** `VS_ENDPOINT` as PDF chunks (or a dedicated one). Uncomment `create_delta_sync_index` once per index name.


In [ ]:
%pip install -q databricks-vectorsearch
dbutils.library.restartPython()


In [ ]:
from databricks.vector_search.client import VectorSearchClient

VS_ENDPOINT = "YOUR_VS_ENDPOINT_NAME"
INDEX_CURATED = "ayurveda_lakehouse.ayurgenix.ayurgenix_curated_index"
SOURCE_CURATED = "ayurveda_lakehouse.ayurgenix.curated_rows_for_vector"
EMBED_MODEL = "databricks-qwen3-embedding-0-6b"

vsc = VectorSearchClient(disable_notice=True)

# vsc.create_delta_sync_index(
#     endpoint_name=VS_ENDPOINT,
#     index_name=INDEX_CURATED,
#     source_table_name=SOURCE_CURATED,
#     pipeline_type="TRIGGERED",
#     primary_key="curated_pk",
#     embedding_source_column="embedding_text",
#     embedding_model_endpoint_name=EMBED_MODEL,
# )

idx = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_CURATED)
idx.sync()
print("Sync requested for", INDEX_CURATED)


## Step 3 — Smoke test (symptom-style query)


In [ ]:
from databricks.vector_search.client import VectorSearchClient

VS_ENDPOINT = "YOUR_VS_ENDPOINT_NAME"
INDEX_CURATED = "ayurveda_lakehouse.ayurgenix.ayurgenix_curated_index"

cols = [
    "curated_pk",
    "Disease",
    "Symptoms",
    "Ayurvedic_Herbs",
    "Formulation",
    "Doshas",
    "Diet_and_Lifestyle_Recommendations",
    "Patient_Recommendations",
]

vsc = VectorSearchClient(disable_notice=True)
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_CURATED)
print(
    index.similarity_search(
        query_text="cough sore throat chest congestion",
        columns=cols,
        num_results=5,
        query_type="HYBRID",
    )
)
